# Data Exercise 3 – Wind Resource Assessment (Part 3)

---

## Background

Aeolien Industries has reviewed your updated assessment and has one final set of requests before the report is complete:

1. Calculate the **total energy** the turbine can be expected to produce over the month, by combining the wind power curve with a histogram of observed wind speeds
2. Produce a **table and figure of average daily wind speed**
3. Plot the **average wind speed by hour of day** — the kind of display you'd see in a weather app
4. To make step 3 possible, **decompose the decimal-day timestamp** into separate integer columns for day and hour

This notebook introduces three new concepts that are widely used in data science:
- **Histograms** — counting how many observations fall into each value range (bin)
- **GroupBy** — splitting a dataset into groups (by day, by hour, etc.) and computing statistics within each group
- **Floor division** — an arithmetic trick for decomposing decimal timestamps into their component parts

---

## Step 0 – Imports, data, and functions from Parts 1 & 2

We start by setting up everything we built in the previous notebooks. The functions `log_law()` and `wind_power()` are reproduced here so this notebook is self-contained.

**Your turn:** You've done the data loading and `Day` column creation twice now — fill in the two blanks without looking back.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline
print("Libraries loaded!")

In [ ]:
# URL to the excel file we'll be opening
url = "https://github.com/radryan1979/clim341/raw/refs/heads/main/Data/HW1_MtToms_data.xlsx"

# TODO: Load the Excel data file into a DataFrame called df
# Hint: use pd.read_excel() — same as Part 1, the url variable is already provided above

# TODO: Load the data and create the Day column
df = pd.read_excel(___)
df['Day'] = df['Hour'] / ___

print(f"Loaded {len(df)} rows.")
df.head()

In [ ]:
# Functions carried over from Part 2
# These are defined here so this notebook works on its own

def log_law(u_station, z_hub, z_station=2.5, z0=0.5):
    """Extrapolate wind speed from sensor height to hub height using the log law."""
    return u_station * (np.log(z_hub / z0) / np.log(z_station / z0))


def wind_power(u, rotor_diameter, rho=1.225):
    """Calculate available wind power (W) using P = 0.5 * rho * A * U^3."""
    A = np.pi * (rotor_diameter / 2)**2
    return 0.5 * rho * A * u**3


# Apply the log law to get 100 m hub-height wind speeds
df['u_100m'] = log_law(df['m/s'], z_hub=100)

print("Functions defined. Hub-height wind speeds calculated.")
print(f"Mean wind speed at 100 m: {df['u_100m'].mean():.2f} m/s")

---

## Question 1 – Total energy production

### The concept: binning wind speed data

Up until now, we've worked with wind speed as a continuous time series — every 10-minute observation individually. But for estimating total energy production, we want to know: **how often does the wind blow at each speed?**

To answer this, we divide the wind speed range into intervals called **bins** and count how many observations fall into each one. The result is a **histogram** — one of the most fundamental tools in data analysis.

For example, with 0.5 m/s bins:
- Bin 1: 0–0.5 m/s → maybe 12 observations
- Bin 2: 0.5–1.0 m/s → maybe 47 observations
- Bin 3: 1.0–1.5 m/s → maybe 83 observations
- ... and so on

### The energy equation

Once we have the histogram, we can calculate the total energy contribution from each bin:

$$E_i = P(U_i) \times \Delta t_i$$

where:
- $P(U_i)$ = wind power at the **midpoint** wind speed of bin $i$ (Watts)
- $\Delta t_i$ = total time wind spent in bin $i$ = (number of observations in bin) × (10 minutes per observation)

Summing over all bins gives the total energy for the month:

$$E_{\text{total}} = \sum_{i=1}^{N} E_i$$

The units work out as: W × min → W × hr/60 → Wh → kWh (dividing by 1000).

### Step 1: Build the histogram with `np.histogram()`

`np.histogram()` does the counting for us. It returns two arrays:
- `counts` — the number of observations in each bin
- `bin_edges` — the boundaries of each bin

If we have `N` bins, we get `N` counts but `N+1` edges (one for each left and right boundary). The **midpoint** of each bin is the average of its left and right edge.

In [ ]:
# Define the bin edges: 0 to 26 m/s in 0.5 m/s steps
# We go a bit beyond 25 m/s to catch any values at exactly 25 m/s
bin_edges = np.arange(0, 26.5, 0.5)

# np.histogram() returns (counts, edges)
counts, edges = np.histogram(df['u_100m'], bins=bin_edges)

# Calculate the midpoint of each bin
# If edges = [0, 0.5, 1.0, ...], midpoints = [0.25, 0.75, 1.25, ...]
bin_midpoints = (edges[:-1] + edges[1:]) / 2   # average of left and right edge for each bin

print(f"Number of bins:  {len(counts)}")
print(f"Total counts:    {counts.sum()}  (should equal {len(df)})")
print(f"\nFirst 5 bins:")
print(f"  {'Midpoint (m/s)':>16}  {'Count':>6}")
for mid, cnt in zip(bin_midpoints[:5], counts[:5]):
    print(f"  {mid:>16.2f}  {cnt:>6}")

### Step 2: Visualize the histogram

Before calculating energy, let's look at the wind speed distribution. This is called a **wind speed frequency distribution** — a standard figure in any wind resource report.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

# ax.bar() draws a bar chart; we use bin_midpoints as the x positions
# width=0.45 makes the bars slightly narrower than the 0.5 m/s bin width
ax.bar(bin_midpoints, counts, width=0.45, color='steelblue', edgecolor='white', linewidth=0.5)

ax.set_xlabel('Wind Speed (m/s)',             fontsize=12)
ax.set_ylabel('Number of Observations',       fontsize=12)
ax.set_title('Wind Speed Frequency Distribution at 100 m Hub Height', fontsize=13)

plt.tight_layout()
plt.show()

### Step 3: Calculate energy per bin

Now we apply the energy equation to every bin at once. This is another example of vectorized computation — instead of looping through bins one by one, we pass the entire `bin_midpoints` array to `wind_power()` and it returns an array of power values.

The time step is 10 minutes per observation. We'll convert to hours for kWh.

In [ ]:
# Time step: each observation represents 10 minutes
dt_minutes = 10
dt_hours   = dt_minutes / 60    # convert to hours for kWh

# Step 1: Calculate power (W) at the midpoint of each bin
P_per_bin_W = wind_power(u=bin_midpoints, rotor_diameter=126)

# Step 2: Total time in each bin (hours) = count × time per observation
dt_per_bin_hours = counts * dt_hours

# Step 3: Energy per bin (Wh) = Power × time
E_per_bin_Wh = P_per_bin_W * dt_per_bin_hours

# Step 4: Convert to kWh and sum over all bins
E_per_bin_kWh  = E_per_bin_Wh / 1000
E_total_kWh    = E_per_bin_kWh.sum()
E_total_MWh    = E_total_kWh / 1000

print("=" * 45)
print("  Monthly Energy Production Estimate")
print("=" * 45)
print(f"  Total energy: {E_total_kWh:>12,.0f} kWh")
print(f"  Total energy: {E_total_MWh:>12,.1f} MWh")
print("=" * 45)

### Step 4: Visualize energy contribution by bin

It's instructive to see *which wind speeds contribute the most energy*. Low wind speeds have many observations but low power. Very high wind speeds have high power but are rare. The dominant contribution comes from somewhere in between.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left panel: wind speed frequency (how often each speed occurs)
axes[0].bar(bin_midpoints, counts, width=0.45, color='steelblue', edgecolor='white', linewidth=0.5)
axes[0].set_xlabel('Wind Speed (m/s)',       fontsize=11)
axes[0].set_ylabel('Number of Observations', fontsize=11)
axes[0].set_title('Wind Speed Distribution', fontsize=12)

# Right panel: energy contribution per bin
axes[1].bar(bin_midpoints, E_per_bin_kWh, width=0.45, color='tomato', edgecolor='white', linewidth=0.5)
axes[1].set_xlabel('Wind Speed (m/s)', fontsize=11)
axes[1].set_ylabel('Energy (kWh)',     fontsize=11)
axes[1].set_title('Energy Contribution by Wind Speed Bin', fontsize=12)

plt.suptitle('Gamesa G126-2.625 MW at Mt. Tom (100 m hub height)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

**✏️ Your answer:**

> *Report the total estimated monthly energy output in kWh and MWh. Compare the two panels: which wind speeds are most common? Which wind speeds contribute the most energy? Explain the difference.*

---

## Questions 2 & 3 – Average wind speed by day and by hour

### Introducing GroupBy

The next two questions both follow the same pattern:
- Take the full dataset of 4,464 observations
- **Group** the rows by some label (day number, or hour of day)
- **Compute a statistic** within each group (here, the mean wind speed)

This operation — split into groups, apply a function, collect results — is called **GroupBy**, and pandas makes it very clean:

```python
df.groupby('group_column')['value_column'].mean()
```

That single line replaces what would otherwise be a complex loop. It returns a new Series where the index is the group label and the values are the group means.

But before we can group by *day* or *hour*, we need those as **separate integer columns** in the DataFrame. The `Day` column we created in Part 1 is a *decimal* number — `1.16667` means day 1, at hour 4. We need to pull apart the whole-number part (the day) from the fractional part (the time within the day).

We'll tackle that decomposition in Question 4. For now, let's handle **daily averages** first — that one only requires knowing which integer day each row belongs to, which we can get directly from `Day` with a simple rounding-down operation.

---

## Question 4 – Decomposing the timestamp

### Floor division and the modulo operator

To break a decimal day like `3.7083` into integer day `3` and integer hour `17`, we use two arithmetic operations you may not have seen in Python yet:

| Operation | Python symbol | Example | Result |
|---|---|---|---|
| **Floor division** — divide and round down to integer | `//` | `3.7083 // 1` | `3.0` |
| **Modulo** — the remainder after division | `%` | `3.7083 % 1` | `0.7083` |

Floor division is how we extract the day. The fractional remainder (`0.7083`) represents the fraction of the day elapsed — multiply by 24 to get the hour, then floor again:

```
Day  = 3.7083
  integer day   = 3.7083 // 1         = 3       (floor division by 1 drops the decimal)
  day fraction  = 3.7083 % 1          = 0.7083  (what's left over)
  decimal hour  = 0.7083 * 24         = 17.0
  integer hour  = (0.7083 * 24) // 1  = 17
```

Because pandas applies arithmetic operations to every row at once, we can compute entire new columns in a single line.

In [ ]:
# Integer day: floor-divide the decimal day by 1
# e.g. 3.7083 // 1 = 3.0  →  we cast to int for cleanliness
df['day_int'] = (df['Day'] // 1).astype(int)

# Integer hour: take the fractional part of the day, multiply by 24, then floor
df['hour_int'] = ((df['Day'] % 1) * 24 // 1).astype(int)

# Preview a few rows to verify the decomposition makes sense
print("Spot-check: decimal Day → integer day and hour")
df[['Day', 'day_int', 'hour_int', 'm/s']].iloc[0:12:2]  # every other row

Check the numbers manually on a couple of rows — does the integer day and hour match what you'd expect from the decimal day value?

---

### Extra Credit – Integer minute column

Using the same logic, we can go one step further and extract the integer minute from the timestamp. The minute is the fractional part of the decimal hour, multiplied by 60.

**Your turn:** Fill in the expression below. Hint: the pattern is identical to how we got `hour_int` from `Day`, but applied one level deeper.

In [ ]:
# Extra credit: integer minute column
# Step 1: decimal hour = fractional part of Day × 24
decimal_hour = (df['Day'] % 1) * 24

# Step 2: TODO - extract integer minute from the fractional part of decimal_hour
# Hint: same as how hour_int was extracted from Day, but multiply by 60 not 24
df['minute_int'] = ((___ % 1) * ___ // 1).astype(int)

# Verify
print("Decomposed timestamp check:")
df[['Hour', 'day_int', 'hour_int', 'minute_int']].head(12)

---

## Question 2 – Average wind speed per day

With `day_int` in hand, we can now use **groupby** to compute the mean wind speed for each day of the month.

The result is a pandas **Series** — a single column of values with the day number as the index. We can display it as a table directly, and pass it straight to the plotter.

In [ ]:
# Group all rows by their integer day label, then compute the mean wind speed within each group
daily_mean = df.groupby('day_int')['u_100m'].mean()

print(f"Number of days in dataset: {len(daily_mean)}")
print(f"Grand mean across all days: {daily_mean.mean():.2f} m/s")

In [ ]:
# Display as a formatted table
daily_table = daily_mean.reset_index()
daily_table.columns = ['Day of Month', 'Mean Wind Speed (m/s)']
daily_table['Mean Wind Speed (m/s)'] = daily_table['Mean Wind Speed (m/s)'].round(2)

print("Average wind speed by day:")
print(daily_table.to_string(index=False))

In [ ]:
# Plot average daily wind speed
# TODO: Fill in the axis labels and title
fig, ax = plt.subplots(figsize=(10, 4))

ax.bar(daily_mean.index, daily_mean.values,
       color='steelblue', edgecolor='white', linewidth=0.5)

# Add a horizontal dashed line at the overall mean for reference
ax.axhline(y=daily_mean.mean(), color='tomato', linestyle='--',
           linewidth=1.5, label=f'Monthly mean ({daily_mean.mean():.2f} m/s)')

ax.set_xlabel('___', fontsize=12)
ax.set_ylabel('___', fontsize=12)
ax.set_title('___',  fontsize=13)
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

**✏️ Your answer:**

> *Which day had the highest average wind speed? Which had the lowest? Is there any apparent pattern — are windy and calm days clustered together, or do they seem random?*

---

## Question 3 – Average wind speed by hour of day

This figure answers the question: **on a typical day, when is the wind strongest?** Weather apps often show this kind of plot — it collapses a full month of data into a single "average day" by grouping all observations that occurred at the same hour across all days.

The logic is identical to the daily average above — just replace `day_int` with `hour_int`.

**Your turn:** The cell below is mostly blank. Use the daily average code above as your template — you're doing the same thing but grouping by `hour_int` instead of `day_int`.

In [ ]:
# TODO: Group by hour_int and compute mean wind speed
# Follow the exact same pattern as daily_mean above
hourly_mean = df.groupby('___')['u_100m'].___

print(f"Hours in dataset: {len(hourly_mean)}")
print(hourly_mean)

In [ ]:
# TODO: Plot average wind speed by hour of day
# Use a LINE plot (ax.plot) rather than bars — hourly data is continuous and cyclical
# Include axis labels, a title, and a horizontal reference line for the overall mean
# (Hint: look at the daily plot above for the axhline syntax)

fig, ax = plt.subplots(figsize=(9, 4))

# TODO: plot hourly_mean.index on x, hourly_mean.values on y
ax.plot(___, ___,
        color='steelblue', linewidth=2.0,
        marker='o', markersize=5,
        label='Hourly mean wind speed')

# TODO: add overall mean reference line
ax.axhline(y=___, color='tomato', linestyle='--', linewidth=1.5,
           label=f'Monthly mean ({hourly_mean.mean():.2f} m/s)')

# TODO: Set x-ticks to show every hour clearly
ax.set_xticks(range(0, 24))

ax.set_xlabel('___', fontsize=12)
ax.set_ylabel('___', fontsize=12)
ax.set_title('___',  fontsize=13)
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

**✏️ Your answer:**

> *Describe what you see. When during the day is the wind typically strongest at this site? When is it weakest? This pattern — wind increasing through the day and decreasing at night — is common in many regions. Can you think of a physical reason why this might happen? (Hint: think about how the sun heats the surface and what that does to the atmosphere.)*

---

## Final summary

Run the cell below to print a complete summary of the full three-part wind resource assessment.

In [ ]:
print("=" * 52)
print("  Mt. Tom – Complete Wind Resource Summary")
print("  Aeolien Industries / Gamesa G126-2.625 MW")
print("=" * 52)
print("  Wind resource (100 m hub height)")
print(f"    Mean wind speed:   {df['u_100m'].mean():.2f} m/s")
print(f"    Std deviation:     {df['u_100m'].std():.2f} m/s")
print(f"    Max wind speed:    {df['u_100m'].max():.2f} m/s")
print(f"    Min wind speed:    {df['u_100m'].min():.2f} m/s")
print("-" * 52)
print("  Turbine operation (Gamesa G126, 1–25 m/s)")
from functools import reduce
op_mask = (df['u_100m'] >= 1) & (df['u_100m'] <= 25)
print(f"    Operational:       {op_mask.sum()/len(df)*100:.1f}% of observations")
print("-" * 52)
print("  Energy production")
print(f"    Monthly energy:    {E_total_kWh:>10,.0f} kWh")
print(f"    Monthly energy:    {E_total_MWh:>10,.1f} MWh")
print("-" * 52)
print("  Diurnal pattern (hourly means, 100 m)")
peak_hour = hourly_mean.idxmax()
calm_hour = hourly_mean.idxmin()
print(f"    Windiest hour:     {peak_hour:02d}:00  ({hourly_mean.max():.2f} m/s)")
print(f"    Calmest hour:      {calm_hour:02d}:00  ({hourly_mean.min():.2f} m/s)")
print("=" * 52)

---

## 🎉 Wind resource assessment complete!

Across the three notebooks, you've built a complete data analysis pipeline from scratch:

| Part | What you learned |
|---|---|
| **1** | Loading data, computing statistics, basic plotting, the log-law |
| **2** | Writing reusable functions, the DRY principle, the power equation |
| **3** | Histograms and binning, energy summation, GroupBy aggregation, timestamp decomposition |

These are the core tools of scientific data analysis in Python — and the same techniques scale directly to real research datasets orders of magnitude larger than this one.